# Reading a CSV

In [1]:
import Data.Csv (FromRecord(..), ToRecord, HasHeader(..))
import GHC.Generics
import System.IO
import System.Exit (exitFailure)
-- from bytestring
import Data.ByteString (ByteString, hGetSome, empty)
import qualified Data.ByteString.Lazy as BL
import Data.Csv.Incremental (Parser(..), decode)

In [2]:
import Data.Either (lefts, rights)

In [3]:
import Numeric.LinearAlgebra ((<\>), takeDiag, inv, tr, dot, (<#), ident, (!), svd)
import qualified Numeric.LinearAlgebra as LA ((<>))
import Numeric.LinearAlgebra.Data (fromLists, fromList, Vector, rows)

In [4]:
data SpectorRecord = SpectorRecord
  { obs :: !Int
  , gpa  :: !Double
  , tuce :: !Int
  , psi :: !Int
  , grade :: !Int
  } deriving (Show, Eq, Generic)

instance FromRecord SpectorRecord
instance ToRecord SpectorRecord


In [5]:
:info parseRecord

type FromRecord :: * -> Constraint
class FromRecord a where
  parseRecord :: cassava-0.5.3.0-KyZsaqzYvt6KMp6f7j98eP:Data.Csv.Types.Record -> cassava-0.5.3.0-KyZsaqzYvt6KMp6f7j98eP:Data.Csv.Conversion.Parser a
  default parseRecord :: (Generic a, cassava-0.5.3.0-KyZsaqzYvt6KMp6f7j98eP:Data.Csv.Conversion.GFromRecord (Rep a)) =>
                         cassava-0.5.3.0-KyZsaqzYvt6KMp6f7j98eP:Data.Csv.Types.Record -> cassava-0.5.3.0-KyZsaqzYvt6KMp6f7j98eP:Data.Csv.Conversion.Parser a
  	-- Defined in ‘cassava-0.5.3.0-KyZsaqzYvt6KMp6f7j98eP:Data.Csv.Conversion’

In [6]:
feed :: (ByteString -> Parser SpectorRecord) -> Handle -> IO (Parser SpectorRecord)
feed k csvFile = do
  hIsEOF csvFile >>= \eof -> case eof of
    True  -> return $ k empty
    False -> k <$> hGetSome csvFile 4096

spector <- withFile "spector.csv" ReadMode $ \ csvFile -> do
  let loop !_ (Fail _ errMsg) = do putStrLn errMsg; return []
      loop acc (Many rs k)    = loop (acc <> rs) =<< feed k csvFile
      loop acc (Done rs)      = return $ (acc <> rs)
  loop [] (decode HasHeader)


In [7]:
rights spector

[SpectorRecord {obs = 1, gpa = 2.66, tuce = 20, psi = 0, grade = 0},SpectorRecord {obs = 2, gpa = 2.89, tuce = 22, psi = 0, grade = 0},SpectorRecord {obs = 3, gpa = 3.28, tuce = 24, psi = 0, grade = 0},SpectorRecord {obs = 4, gpa = 2.92, tuce = 12, psi = 0, grade = 0},SpectorRecord {obs = 5, gpa = 4.0, tuce = 21, psi = 0, grade = 1},SpectorRecord {obs = 6, gpa = 2.86, tuce = 17, psi = 0, grade = 0},SpectorRecord {obs = 7, gpa = 2.76, tuce = 17, psi = 0, grade = 0},SpectorRecord {obs = 8, gpa = 2.87, tuce = 21, psi = 0, grade = 0},SpectorRecord {obs = 9, gpa = 3.03, tuce = 25, psi = 0, grade = 0},SpectorRecord {obs = 10, gpa = 3.92, tuce = 29, psi = 0, grade = 1},SpectorRecord {obs = 11, gpa = 2.63, tuce = 20, psi = 0, grade = 0},SpectorRecord {obs = 12, gpa = 3.32, tuce = 23, psi = 0, grade = 0},SpectorRecord {obs = 13, gpa = 3.57, tuce = 23, psi = 0, grade = 0},SpectorRecord {obs = 14, gpa = 3.26, tuce = 25, psi = 0, grade = 1},SpectorRecord {obs = 15, gpa = 3.53, tuce = 26, psi = 0, 

In [8]:
x0 = fromLists $ map (\spector -> [1.0, gpa spector, fromIntegral $ tuce spector, fromIntegral $ psi spector]) (rights spector)
y = fromList $ map (fromIntegral . grade) (rights spector) :: Vector Double

In [9]:
import Numeric.LinearAlgebra (thinSVD, disp, diag, diagRect)
import Numeric.LinearAlgebra.Data (rows, cols)

In [10]:
import Numeric.LinearAlgebra.Data (maxElement, cmap, subVector, conj)
import Numeric.LinearAlgebra (Matrix)

In [27]:
-- pinv_extended :: Matrix Double -> Double -> ?
-- svd :: Field t => Matrix t -> (Matrix t, Vector Double, Matrix t)
let (u, s, v) = svd $ conj x0
disp 2 u
disp 2 $ diagRect 0 s 4 32
disp 2 v

32x32
-0.16  -0.04  -0.14   0.20  -0.19  -0.20  -0.20  -0.17  -0.13  -0.11  -0.17  -0.16  -0.16  -0.14  -0.13  -0.18  -0.13  -0.18  -0.19  -0.17  -0.18  -0.15  -0.27  -0.17  -0.19  -0.15  -0.25  -0.17  -0.22  -0.21  -0.21  -0.21
-0.17  -0.06  -0.15   0.09   0.33  -0.09  -0.13  -0.11  -0.06   0.27  -0.20   0.06   0.16   0.03   0.13  -0.15  -0.17  -0.11   0.02   0.03  -0.39   0.19  -0.03   0.16   0.18  -0.11   0.15  -0.16   0.24   0.36   0.02  -0.25
-0.19  -0.04  -0.17  -0.06   0.00   0.11   0.10  -0.08  -0.25  -0.38  -0.05  -0.14  -0.12  -0.24  -0.27   0.01  -0.27   0.01   0.03  -0.06   0.00  -0.17   0.44  -0.08   0.01  -0.18   0.33  -0.05   0.16   0.10   0.12   0.17
-0.10   0.40  -0.12   0.28  -0.24  -0.22  -0.21  -0.13  -0.05  -0.04  -0.13  -0.12  -0.15  -0.07  -0.07  -0.16  -0.02  -0.17   0.17   0.21   0.25   0.24  -0.02   0.20   0.15   0.29  -0.00   0.24   0.07   0.08   0.12   0.15
-0.17   0.32  -0.18  -0.21   0.80  -0.05  -0.04  -0.01   0.01  -0.09   0.02  -0.06  -0.10  -0.03  -0.0

4x32
127.35  0.00  0.00  0.00  0.00  0.00  0.00  0.00  0.00  0.00  0.00  0.00  0.00  0.00  0.00  0.00  0.00  0.00  0.00  0.00  0.00  0.00  0.00  0.00  0.00  0.00  0.00  0.00  0.00  0.00  0.00  0.00
  0.00  3.19  0.00  0.00  0.00  0.00  0.00  0.00  0.00  0.00  0.00  0.00  0.00  0.00  0.00  0.00  0.00  0.00  0.00  0.00  0.00  0.00  0.00  0.00  0.00  0.00  0.00  0.00  0.00  0.00  0.00  0.00
  0.00  0.00  2.79  0.00  0.00  0.00  0.00  0.00  0.00  0.00  0.00  0.00  0.00  0.00  0.00  0.00  0.00  0.00  0.00  0.00  0.00  0.00  0.00  0.00  0.00  0.00  0.00  0.00  0.00  0.00  0.00  0.00
  0.00  0.00  0.00  0.73  0.00  0.00  0.00  0.00  0.00  0.00  0.00  0.00  0.00  0.00  0.00  0.00  0.00  0.00  0.00  0.00  0.00  0.00  0.00  0.00  0.00  0.00  0.00  0.00  0.00  0.00  0.00  0.00

4x4
-0.04   0.20  -0.01   0.98
-0.14   0.96  -0.08  -0.21
-0.99  -0.15  -0.01  -0.01
-0.02   0.08   1.00  -0.01

In [28]:
tr x0

(4><32)
 [  1.0,  1.0,  1.0,  1.0,  1.0,  1.0,  1.0,  1.0,  1.0,  1.0,  1.0,  1.0,  1.0,  1.0,  1.0,  1.0,  1.0,  1.0,  1.0,  1.0,  1.0,  1.0,  1.0,  1.0,  1.0,  1.0,  1.0,  1.0,  1.0,  1.0,  1.0,  1.0
 , 2.66, 2.89, 3.28, 2.92,  4.0, 2.86, 2.76, 2.87, 3.03, 3.92, 2.63, 3.32, 3.57, 3.26, 3.53, 2.74, 2.75, 2.83, 3.12, 3.16, 2.06, 3.62, 2.89, 3.51, 3.54, 2.83, 3.39, 2.67, 3.65,  4.0,  3.1, 2.39
 , 20.0, 22.0, 24.0, 12.0, 21.0, 17.0, 17.0, 21.0, 25.0, 29.0, 20.0, 23.0, 23.0, 25.0, 26.0, 19.0, 25.0, 19.0, 23.0, 25.0, 22.0, 28.0, 14.0, 26.0, 24.0, 27.0, 17.0, 24.0, 21.0, 23.0, 21.0, 19.0
 ,  0.0,  0.0,  0.0,  0.0,  0.0,  0.0,  0.0,  0.0,  0.0,  0.0,  0.0,  0.0,  0.0,  0.0,  0.0,  0.0,  0.0,  0.0,  1.0,  1.0,  1.0,  1.0,  1.0,  1.0,  1.0,  1.0,  1.0,  1.0,  1.0,  1.0,  1.0,  1.0 ]

In [29]:
disp 4 $ u <> diagRect 0 s 32 4 <> tr v

32x4
1.0000  2.6600  20.0000  0.0000
1.0000  2.8900  22.0000  0.0000
1.0000  3.2800  24.0000  0.0000
1.0000  2.9200  12.0000  0.0000
1.0000  4.0000  21.0000  0.0000
1.0000  2.8600  17.0000  0.0000
1.0000  2.7600  17.0000  0.0000
1.0000  2.8700  21.0000  0.0000
1.0000  3.0300  25.0000  0.0000
1.0000  3.9200  29.0000  0.0000
1.0000  2.6300  20.0000  0.0000
1.0000  3.3200  23.0000  0.0000
1.0000  3.5700  23.0000  0.0000
1.0000  3.2600  25.0000  0.0000
1.0000  3.5300  26.0000  0.0000
1.0000  2.7400  19.0000  0.0000
1.0000  2.7500  25.0000  0.0000
1.0000  2.8300  19.0000  0.0000
1.0000  3.1200  23.0000  1.0000
1.0000  3.1600  25.0000  1.0000
1.0000  2.0600  22.0000  1.0000
1.0000  3.6200  28.0000  1.0000
1.0000  2.8900  14.0000  1.0000
1.0000  3.5100  26.0000  1.0000
1.0000  3.5400  24.0000  1.0000
1.0000  2.8300  27.0000  1.0000
1.0000  3.3900  17.0000  1.0000
1.0000  2.6700  24.0000  1.0000
1.0000  3.6500  21.0000  1.0000
1.0000  4.0000  23.0000  1.0000
1.0000  3.1000  21.0000  1.0000
1.0

In [14]:
disp 2 $ v <> tr v

4x4
 1.00  -0.00  0.00  -0.00
-0.00   1.00  0.00  -0.00
 0.00   0.00  1.00   0.00
-0.00  -0.00  0.00   1.00

In [26]:
-- pinv_extended :: Matrix Double -> Double -> ?
let (u, s, v) = thinSVD $ conj x0
disp 4 u
disp 4 $ diagRect 0 s 4 4
disp 4 v

32x4
-0.1586  -0.0411  -0.1381   0.1964
-0.1744  -0.0625  -0.1505   0.0914
-0.1903  -0.0354  -0.1677  -0.0592
-0.0967   0.4015  -0.1236   0.2787
-0.1678   0.3189  -0.1809  -0.2063
-0.1355   0.1559  -0.1357   0.1979
-0.1354   0.1256  -0.1327   0.2265
-0.1666  -0.0230  -0.1472   0.1167
-0.1978  -0.1566  -0.1631  -0.0073
-0.2299  -0.0692  -0.2008  -0.3401
-0.1586  -0.0502  -0.1372   0.2049
-0.1826   0.0222  -0.1662  -0.0511
-0.1829   0.0978  -0.1736  -0.1226
-0.1981  -0.0870  -0.1699  -0.0731
-0.2061  -0.0508  -0.1808  -0.1699
-0.1509   0.0286  -0.1377   0.1931
-0.1975  -0.2413  -0.1547   0.0727
-0.1510   0.0558  -0.1404   0.1673
-0.1825  -0.0129   0.1973  -0.0066
-0.1981  -0.0918   0.1906  -0.0572
-0.1736  -0.2882   0.2317   0.3161
-0.2219  -0.0890   0.1685  -0.2474
-0.1124   0.3269   0.2293   0.2355
-0.2063  -0.0314   0.1773  -0.1768
-0.1908   0.0687   0.1820  -0.1462
-0.2133  -0.2826   0.1948  -0.0020
-0.1362   0.3417   0.2060   0.0338
-0.1898  -0.1946   0.2080   0.1025
-0.1676   0.238

4x4
127.3469  0.0000  0.0000  0.0000
  0.0000  3.1889  0.0000  0.0000
  0.0000  0.0000  2.7872  0.0000
  0.0000  0.0000  0.0000  0.7253

4x4
-0.0438   0.2032  -0.0084   0.9781
-0.1378   0.9649  -0.0832  -0.2074
-0.9893  -0.1450  -0.0078  -0.0142
-0.0196   0.0811   0.9965  -0.0092

In [16]:
pinvExtended :: Matrix Double -> Double -> (Matrix Double, Vector Double)
pinvExtended m rcond = (v <> diagRect 0 srecip rrow rcol <> tr u, s)
    where (u, s, v) = thinSVD $ conj x0
          rcol = cols u
          rrow = cols v
          urow = rows u
          vrow = rows v
          cutoff = rcond * maxElement s
          srecip = cmap (\x -> if x > cutoff then 1.0 / x else 0.0) (subVector 0 (min urow vrow) s)

-- print cutoff
-- let srecip = cmap (\x -> if x > cutoff then 1.0 / x else 0.0) (subVector 0 (min (rows u) (rows v)) s)
-- print srecip
-- let ret = v <> diagRect 0 srecip (cols v) (rows u) <> tr u
-- disp 4 u
-- disp 4 $ diagRect 0 s 4 32
-- disp 4 v

In [19]:
let (xinv, s) = pinvExtended x0 1e-15
disp 3 $ xinv <> x0 -- identity
disp 3 $ x0 <> xinv 

4x4
 1.000  0.000   0.000   0.000
-0.000  1.000  -0.000  -0.000
 0.000  0.000   1.000   0.000
-0.000  0.000  -0.000   1.000

32x32
 0.084   0.069   0.043   0.071  -0.002   0.073   0.079   0.071   0.059   0.000   0.086   0.041   0.025   0.044   0.026   0.080   0.077   0.074   0.001  -0.002   0.069  -0.033   0.019  -0.025  -0.026   0.018  -0.014   0.030  -0.032  -0.056   0.003   0.049
 0.069   0.065   0.055   0.036   0.018   0.052   0.056   0.063   0.068   0.044   0.070   0.051   0.041   0.059   0.051   0.063   0.079   0.059   0.002   0.006   0.042  -0.004  -0.014  -0.005  -0.012   0.025  -0.026   0.023  -0.025  -0.033  -0.003   0.020
 0.043   0.055   0.069   0.008   0.063   0.031   0.030   0.050   0.071   0.100   0.043   0.065   0.068   0.074   0.081   0.039   0.068   0.040   0.002   0.012  -0.014   0.032  -0.043   0.021   0.012   0.018  -0.023   0.002  -0.001   0.013  -0.007  -0.025
 0.071   0.036   0.008   0.263   0.109   0.148   0.143   0.058  -0.026  -0.076   0.069   0.033   0.044  -0.015  -0.025   0.097  -0.038   0.101  -0.014  -0.057  -0.039  -0.104   0.179  -0.064  -0.017  -0.117   0.134  -0.057   0.05

In [20]:
-- import Numeric.LinearAlgebra.Data (maxElement, cmap, subVector)
import Numeric.LinearAlgebra.HMatrix (rank, app)
import Numeric.LinearAlgebra (norm_2)

In [22]:
-- TODO: need to figure out the pre-processing to get wexog
let wexog = x0
    wendog = y -- TODO: I think
    (pinvWexog, singularValues) = pinvExtended wexog 1e-15
    normalized_cov_params = pinvWexog <> tr pinvWexog -- TODO: Should this be tr'?
    -- Cache these singular values for use later.
    wexogSingularValues = singularValues
    dataRank = rank $ diag singularValues -- following statsmodels, but is this just the number of non-zero singular values, possibly with a cutoff
    beta = pinvWexog `app` wendog


In [23]:
class Model m x y where
    predict :: m -> x -> y
    
class ModelEstimator f x y r where
    fit :: f -> x -> y -> r

In [24]:
data CovarianceType = NonRobustCovariance
  deriving (Eq, Show)

data OLSResults = OLSResults 
    { olsResults_beta :: Vector Double
    , olsResults_normalizedCovParams :: Matrix Double
    , olsResults_covarianceType :: CovarianceType
    , olsResults_uset :: Bool
    , olsResults_trainingData :: (Matrix Double, Vector Double) } deriving (Eq, Show)


In [25]:
instance Model OLSResults (Matrix Double) (Vector Double) where
    predict olsres x = x `app` olsResults_beta olsres

In [26]:
:info y

y :: Vector Double 	-- Defined at <interactive>:2:1

In [60]:
fitOLS :: Matrix Double -> Vector Double -> OLSResults
fitOLS x y = let 
    wexog = x
    wendog = y
    (pinvWexog, singularValues) = pinvExtended wexog 1e-15
    normalized_cov_params = pinvWexog <> tr pinvWexog -- TODO: Should this be tr'?
    -- Cache these singular values for use later.
    wexogSingularValues = singularValues
    dataRank = rank $ diag singularValues -- following statsmodels, but is this just the number of non-zero singular values, possibly with a cutoff
    beta = pinvWexog `app` wendog
    in OLSResults beta normalized_cov_params NonRobustCovariance True (wexog, wendog)

In [61]:
let ols = fitOLS x0 y
ols

OLSResults {olsResults_beta = [-1.498017120399608,0.46385167930975885,1.0495122237428397e-2,0.3785547879260215], olsResults_normalizedCovParams = (4><4)
 [      1.822579758158066,  -0.3661454321030627,  -2.929947841972226e-2, -1.6522254152328518e-2
 ,    -0.3661454321030627,  0.17418252315505742,   -8.07281099139219e-3,   6.471756098827789e-4
 ,  -2.929947841972226e-2, -8.07281099139219e-3,   2.520657013130186e-3, -1.9039298139220165e-3
 , -1.6522254152328518e-2, 6.471756098827789e-4, -1.9039298139220165e-3,    0.12862250679728307 ], olsResults_covarianceType = NonRobustCovariance, olsResults_uset = True, olsResults_trainingData = ((32><4)
 [ 1.0, 2.66, 20.0, 0.0
 , 1.0, 2.89, 22.0, 0.0
 , 1.0, 3.28, 24.0, 0.0
 , 1.0, 2.92, 12.0, 0.0
 , 1.0,  4.0, 21.0, 0.0
 , 1.0, 2.86, 17.0, 0.0
 , 1.0, 2.76, 17.0, 0.0
 , 1.0, 2.87, 21.0, 0.0
 , 1.0, 3.03, 25.0, 0.0
 , 1.0, 3.92, 29.0, 0.0
 , 1.0, 2.63, 20.0, 0.0
 , 1.0, 3.32, 23.0, 0.0
 , 1.0, 3.57, 23.0, 0.0
 , 1.0, 3.26, 25.0, 0.0
 , 1.0, 3.53, 26

In [62]:
:type (predict ols x0 :: Vector Double)

(predict ols x0 :: Vector Double) :: Vector Double

In [63]:
predict ols x0 :: Vector Double

[-5.426920868708147e-2,7.340692202901986e-2,0.27529932143468244,-1.7628749965971434e-2,0.5777871638254237,7.015760462584999e-3,-3.936940746839091e-2,5.36347662053963e-2,0.16983152384467115,0.6246400073800702,-6.818475906637433e-2,0.28335826636964445,0.3993211861970841,0.2765174100859157,0.412252485736979,-2.765619657972912e-2,3.995305363793875e-2,1.4090454558149114e-2,0.5691427184337143,0.6086870300809615,6.696481612794147e-2,0.8535441692757357,0.3680007320556142,0.7815302400768053,0.7744555459812413,0.4766062203835978,0.6314119384227788,0.3709045849817511,0.7939938639930295,0.977332196226302,0.5388754403726623,0.18855050358787676]

In [31]:
-- import Numeric.LinearAlgebra.Static (mean)
import Numeric.LinearAlgebra.Data (size, konst, scalar)
import Numeric.LinearAlgebra (sumElements, Container)
import Data.Vector (length)

In [32]:
:type konst

konst :: forall e d (c :: * -> *). Konst e d c => e -> d -> c e

In [33]:
konst 7 3 :: Vector Double

[7.0,7.0,7.0]

In [34]:
fromList [1..5] - scalar 5.0

[-4.0,-3.0,-2.0,-1.0,0.0]

In [35]:
(fromIntegral . size) $ fromList [1..5]
sumElements (fromList [1..5]) / (fromIntegral . size) (fromList [1..5])

5

3.0

In [36]:
-- Had to import Container for this.  Not sure if importing length would work too.
mean :: (Container Vector e, Fractional e) => Vector e -> e
mean v | size v > 0 = sumElements v / (fromIntegral . size) v
       | otherwise = 0
-- mean v | vsize > 0 = sumElements v / (fromIntegral . size) v
--        | otherwise = 0.0
--     where vsize = size v :: Int

In [37]:
nobs :: OLSResults -> Int
nobs ols = let
    (wexog, _) = olsResults_trainingData ols
    in rows wexog

    
fittedvalues :: OLSResults -> Vector Double
fittedvalues ols = let
    (wexog, _) = olsResults_trainingData ols
    in predict ols wexog

-- NOTE: For OLS, we should consider dropping wexog, wendog, wresid, etc.
wresid :: OLSResults -> Vector Double
wresid ols = let
    (wexog, wendog) = olsResults_trainingData ols
    in wendog - predict ols wexog

ssr :: OLSResults -> Double
ssr = norm_2 . wresid

centeredTSS :: OLSResults -> Double
centeredTSS ols = let
    (_, wendog) = olsResults_trainingData ols
    r = size wendog
    m = mean wendog
    in norm_2 $ wendog - konst m r

uncenteredTSS :: OLSResults -> Double
uncenteredTSS ols = norm_2 wendog
    where (_, wendog) = olsResults_trainingData ols

-- TODO: The boolean is 
rsquared :: OLSResults -> Bool -> Double
rsquared ols hasconst = if hasconst
    then 1.0 - ssr ols / centeredTSS ols
    else 1.0 - ssr ols / uncenteredTSS ols

-- The explained sum of squares.
-- If a constant is present, the centered total sum of squares minus the
-- sum of squared residuals. If there is no constant, the uncentered total
-- sum of squares is used.
ess :: OLSResults -> Bool -> Double
ess ols hasconst = if hasconst
    then centeredTSS ols - ssr ols
    else uncenteredTSS ols - ssr ols


In [ ]:
-- TODO: Some additional work is needed here.
-- Adjusted R-squared.
-- This is defined here as 1 - (`nobs`-1)/`df_resid` * (1-`rsquared`)
-- if a constant is included and 1 - `nobs`/`df_resid` * (1-`rsquared`) if
-- no constant is included.
-- rsquared_adj :: OLSResults -> Bool -> Double
-- rsquared_adj ols hasconst = 1 - adj * (1 - rsquared ols hasconst)
--     where k_constant = if hasconst then 1 else 0
--           adj = (fromIntegral (nobs ols - k_constant) / fromIntegral df_resid)
--           dataRank = rank $ diag singularValues -- TODO
--           df_resid = nobs ols - dataRank

In [38]:
df_resid

: 

In [39]:
rsquared ols True

0.23573545970602305

In [40]:
ess ols True

0.6333677054150968

In [41]:
:info predict

type Model :: * -> * -> * -> Constraint
class Model m x y where
  predict :: m -> x -> y
  	-- Defined at <interactive>:2:5

In [59]:
disp 2 normalized_cov_params
print beta

4x4
 1.82  -0.37  -0.03  -0.02
-0.37   0.17  -0.01   0.00
-0.03  -0.01   0.00  -0.00
-0.02   0.00  -0.00   0.13

[-1.498017120399608,0.46385167930975885,1.0495122237428397e-2,0.3785547879260215]

In [44]:
let k_constant = 1 -- based on the way we defined x0
    nobs = rows wexog
    df_model = dataRank - k_constant
    df_resid = nobs - dataRank

In [58]:
print nobs
print dataRank
print df_model
print df_resid

32

4

3

28

In [46]:
-- OLS constructor
--     fields for WLS:
--     what's going on with the kwarg check for offset?

-- def __init__(self, endog, exog=None, missing='none', hasconst=None,
--              **kwargs):
--     if "weights" in kwargs:
--         msg = ("Weights are not supported in OLS and will be ignored"
--                "An exception will be raised in the next version.")
--         warnings.warn(msg, ValueWarning)
--     super().__init__(endog, exog, missing=missing,
--                               hasconst=hasconst, **kwargs)
--     if "weights" in self._init_keys:
--         self._init_keys.remove("weights")

--     if type(self) is OLS:
--         self._check_kwargs(kwargs, ["offset"])

-- WLS constructor
--     fields for RegressionModel
--      
-- def __init__(self, endog, exog, weights=1., missing='none', hasconst=None,
--              **kwargs):
--     if type(self) is WLS:
--         self._check_kwargs(kwargs)
--     weights = np.array(weights)
--     if weights.shape == ():
--         if (missing == 'drop' and 'missing_idx' in kwargs and
--                 kwargs['missing_idx'] is not None):
--             # patsy may have truncated endog
--             weights = np.repeat(weights, len(kwargs['missing_idx']))
--         else:
--             weights = np.repeat(weights, len(endog))
--     # handle case that endog might be of len == 1
--     if len(weights) == 1:
--         weights = np.array([weights.squeeze()])
--     else:
--         weights = weights.squeeze()
--     super().__init__(endog, exog, missing=missing,
--                               weights=weights, hasconst=hasconst, **kwargs)
--     nobs = self.exog.shape[0]
--     weights = self.weights
--     if weights.size != nobs and weights.shape[0] != nobs:
--         raise ValueError('Weights must be scalar or same length as design')

-- RegressionModel constructor
--     fields for LikelihoodModel
--     pinv_wexog defaulted to None
--     fields defined in initialize():
--         wexog
--         wendog
--         nobs
--         _df_model -- set to None
--         _df_resid -- set to None
--         rank      -- set to None
--     
-- def __init__(self, endog, exog, **kwargs):
--     super().__init__(endog, exog, **kwargs)
--     self.pinv_wexog: Float64Array | None = None
--     self._data_attr.extend(['pinv_wexog', 'wendog', 'wexog', 'weights'])
-- def initialize(self):
--     """Initialize model components."""
--     self.wexog = self.whiten(self.exog)
--     self.wendog = self.whiten(self.endog)
--     # overwrite nobs from class Model:
--     self.nobs = float(self.wexog.shape[0])

--     self._df_model = None
--     self._df_resid = None
--     self.rank = None

-- LikelihoodModel constructor
--     fields from Model, initialize passes so need to determine the method resolution in derived classes

-- def __init__(self, endog, exog=None, **kwargs):
--     super().__init__(endog, exog, **kwargs)
--     self.initialize()

-- Model constructor
--     data
--     k_constant (from data)
--     exog
--     endog
--     _data_attr
--     _init_keys

-- def __init__(self, endog, exog=None, **kwargs):
--     missing = kwargs.pop('missing', 'none')
--     hasconst = kwargs.pop('hasconst', None)
--     self.data = self._handle_data(endog, exog, missing, hasconst,
--                                   **kwargs)
--     self.k_constant = self.data.k_constant
--     self.exog = self.data.exog
--     self.endog = self.data.endog
--     self._data_attr = []
--     self._data_attr.extend(['exog', 'endog', 'data.exog', 'data.endog'])
--     if 'formula' not in kwargs:  # will not be able to unpickle without these
--         self._data_attr.extend(['data.orig_endog', 'data.orig_exog'])
--     # store keys for extras if we need to recreate model instance
--     # we do not need 'missing', maybe we need 'hasconst'
--     self._init_keys = list(kwargs.keys())
--     if hasconst is not None:
--         self._init_keys.append('hasconst')



In [ ]:
        -- if method == "pinv":
        --     if not (hasattr(self, 'pinv_wexog') and
        --             hasattr(self, 'normalized_cov_params') and
        --             hasattr(self, 'rank')):

        --         self.pinv_wexog, singular_values = pinv_extended(self.wexog)
        --         self.normalized_cov_params = np.dot(
        --             self.pinv_wexog, np.transpose(self.pinv_wexog))

        --         # Cache these singular values for use later.
        --         self.wexog_singular_values = singular_values
        --         self.rank = np.linalg.matrix_rank(np.diag(singular_values))

        --     beta = np.dot(self.pinv_wexog, self.wendog)

        -- elif method == "qr":
        --     if not (hasattr(self, 'exog_Q') and
        --             hasattr(self, 'exog_R') and
        --             hasattr(self, 'normalized_cov_params') and
        --             hasattr(self, 'rank')):
        --         Q, R = np.linalg.qr(self.wexog)
        --         self.exog_Q, self.exog_R = Q, R
        --         self.normalized_cov_params = np.linalg.inv(np.dot(R.T, R))

        --         # Cache singular values from R.
        --         self.wexog_singular_values = np.linalg.svd(R, 0, 0)
        --         self.rank = np.linalg.matrix_rank(R)
        --     else:
        --         Q, R = self.exog_Q, self.exog_R
        --     # Needed for some covariance estimators, see GH #8157
        --     self.pinv_wexog = np.linalg.pinv(self.wexog)
        --     # used in ANOVA
        --     self.effects = effects = np.dot(Q.T, self.wendog)
        --     beta = np.linalg.solve(R, effects)
        -- else:
        --     raise ValueError('method has to be "pinv" or "qr"')

        -- if self._df_model is None:
        --     self._df_model = float(self.rank - self.k_constant)
        -- if self._df_resid is None:
        --     self.df_resid = self.nobs - self.rank

        -- if isinstance(self, OLS):
        --     lfit = OLSResults(
        --         self, beta,
        --         normalized_cov_params=self.normalized_cov_params,
        --         cov_type=cov_type, cov_kwds=cov_kwds, use_t=use_t)
        -- else:
        --     lfit = RegressionResults(
        --         self, beta,
        --         normalized_cov_params=self.normalized_cov_params,
        --         cov_type=cov_type, cov_kwds=cov_kwds, use_t=use_t,
        --         **kwargs)
        -- return RegressionResultsWrapper(lfit)

In [ ]:
-- Results Constructors

-- OLSResults -- no __init__

-- RegressionResults
--    see LikelihoodModelResults
--    _cache
--    _wexog_singular_values
--    df_model
--    df_resid
--    cov_type  set if cov_type=='nonrobust'
--    cov_kwds  set if cov_type=='nonrobust'
--    if cov_type!='nonrobust': adjusted use_t and call self.get_robustcov_results
--        this potentially modifies the following fields (note this would seem to overwrite a similar process in LikelihoodModelResults
--            cov_type
--            cov_kwds
--            use_t
--            cov_params_default
--            df_resid_inference
-- 
-- def __init__(self, model, params, normalized_cov_params=None, scale=1.,
--              cov_type='nonrobust', cov_kwds=None, use_t=None, **kwargs):
--     super().__init__(
--         model, params, normalized_cov_params, scale)

--     self._cache = {}
--     if hasattr(model, 'wexog_singular_values'):
--         self._wexog_singular_values = model.wexog_singular_values
--     else:
--         self._wexog_singular_values = None

--     self.df_model = model.df_model
--     self.df_resid = model.df_resid

--     if cov_type == 'nonrobust':
--         self.cov_type = 'nonrobust'
--         self.cov_kwds = {
--             'description': 'Standard Errors assume that the ' +
--             'covariance matrix of the errors is correctly ' +
--             'specified.'}
--         if use_t is None:
--             use_t = True  # TODO: class default
--         self.use_t = use_t
--     else:
--         if cov_kwds is None:
--             cov_kwds = {}
--         if 'use_t' in cov_kwds:
--             # TODO: we want to get rid of 'use_t' in cov_kwds
--             use_t_2 = cov_kwds.pop('use_t')
--             if use_t is None:
--                 use_t = use_t_2
--             # TODO: warn or not?
--         self.get_robustcov_results(cov_type=cov_type, use_self=True,
--                                    use_t=use_t, **cov_kwds)
--     for key in kwargs:
--         setattr(self, key, kwargs[key])

-- LikelihoodModelResults
--    normalized_cov_params
--    scale
--    _use_t set to False
--    use_t     only set if use_t is in kwargs
--    cov_type  set if cov_type is in kwargs and cov_type=='nonrobust'
--    cov_kwds  set if cov_type is in kwargs and cov_type=='nonrobust'
--    if 'cov_type' is in kwargs but cov_type!='nonrobust', then get_robustcov_results is called
--        this potentially modifies the following fields
--            cov_type
--            cov_kwds
--            use_t
--            cov_params_default
--            df_resid_inference

-- def __init__(self, model, params, normalized_cov_params=None, scale=1.,
--              **kwargs):
--     super().__init__(model, params)
--     self.normalized_cov_params = normalized_cov_params
--     self.scale = scale
--     self._use_t = False
--     # robust covariance
--     # We put cov_type in kwargs so subclasses can decide in fit whether to
--     # use this generic implementation
--     if 'use_t' in kwargs:
--         use_t = kwargs['use_t']
--         self.use_t = use_t if use_t is not None else False
--     if 'cov_type' in kwargs:
--         cov_type = kwargs.get('cov_type', 'nonrobust')
--         cov_kwds = kwargs.get('cov_kwds', {})

--         if cov_type == 'nonrobust':
--             self.cov_type = 'nonrobust'
--             self.cov_kwds = {'description': 'Standard Errors assume that the ' +
--                              'covariance matrix of the errors is correctly ' +
--                              'specified.'}
--         else:
--             from statsmodels.base.covtype import get_robustcov_results
--             if cov_kwds is None:
--                 cov_kwds = {}
--             use_t = self.use_t
--             # TODO: we should not need use_t in get_robustcov_results
--             get_robustcov_results(self, cov_type=cov_type, use_self=True,
--                                   use_t=use_t, **cov_kwds)

-- Results
--     __dict__ updated from kwargs
--     _data_attr__ set to empty list
--     _data_in_cache initialized with three values
--     params -- set in initialize using passed in value
--     model  -- set in initialize using passed in model
--     k_constant -- set in initialize using member of model
-- 
-- def __init__(self, model, params, **kwd):
--     self.__dict__.update(kwd)
--     self.initialize(model, params, **kwd)
--     self._data_attr = []
--     # Variables to clear from cache
--     self._data_in_cache = ['fittedvalues', 'resid', 'wresid']

-- def initialize(self, model, params, **kwargs):
--     """
--     Initialize (possibly re-initialize) a Results instance.

--     Parameters
--     ----------
--     model : Model
--         The model instance.
--     params : ndarray
--         The model parameters.
--     **kwargs
--         Any additional keyword arguments required to initialize the model.
--     """
--     self.params = params
--     self.model = model
--     if hasattr(model, 'k_constant'):
--         self.k_constant = model.k_constant
